# Notebook 1 — Chess: Class Design (Bad to Best)

Welcome! This notebook is for someone new to **object-oriented design (OOD)**.
We will design the classes for a chess game step by step.

**What you'll learn**

1. Why "one big class that does everything" is painful.
2. How **polymorphism** (many shapes, one interface) cleans it up.
3. How to split responsibilities: `Piece`, `Board`, `Game`, `Move`.
4. A tiny UML sketch so you can see the relationships.

> **Big idea:** Good OOD is about **who owns what knowledge**.
> A pawn should know how a pawn moves. The board just stores pieces.
> The game just enforces turns and rules. No single class should know everything.


## Setup

```bash
cd 07-object-oriented-design/chess
uv sync
```

In VS Code, click the kernel picker (top-right of the notebook) and pick the `.venv` kernel.
If it doesn't appear: `Cmd+Shift+P` then **Reload Window**.


## Real-world analogy

Think of chess like a **restaurant**:

| Restaurant role | Chess equivalent | Responsibility |
|---|---|---|
| The kitchen     | `Board`            | Holds ingredients (pieces) in known spots |
| Each chef       | Each `Piece` class | Knows how to cook their dish (how to move) |
| The manager     | `Game`             | Enforces rules (whose turn, is the order valid?) |
| The order ticket| `Move`             | Describes "what to do" as data we can log/undo |

If the manager tried to cook every dish personally, the kitchen would be chaos.
That's the bad design we will start with.


## Bad attempt #1 — One giant class with string types

A beginner's first instinct is often: *"I'll just store the piece type as a string
and write one big function with `if/elif` to decide moves."*

It works... until it doesn't. Let's see why.

In [ ]:
# BAD: one monolithic class, piece identity encoded as strings
class BadChess:
    def __init__(self):
        # 8x8 board; each square is None or a tuple ('P', 'W') = white pawn
        self.board = [[None]*8 for _ in range(8)]
        self.board[6][0] = ('P', 'W')   # white pawn
        self.board[7][0] = ('R', 'W')   # white rook
        self.board[0][1] = ('N', 'B')   # black knight

    def valid_moves(self, r, c):
        piece = self.board[r][c]
        if piece is None:
            return []
        kind, color = piece
        moves = []
        if kind == 'P':                 # pawn logic
            dr = -1 if color == 'W' else 1
            if 0 <= r+dr < 8 and self.board[r+dr][c] is None:
                moves.append((r+dr, c))
        elif kind == 'R':               # rook logic
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                while 0 <= nr < 8 and 0 <= nc < 8 and self.board[nr][nc] is None:
                    moves.append((nr, nc)); nr += dr; nc += dc
        elif kind == 'N':               # knight logic
            for dr, dc in [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]:
                nr, nc = r+dr, c+dc
                if 0 <= nr < 8 and 0 <= nc < 8 and self.board[nr][nc] is None:
                    moves.append((nr, nc))
        # ...and this keeps growing: 'B', 'Q', 'K'...
        return moves

bad = BadChess()
print('rook moves:', bad.valid_moves(7, 0)[:5], '...')
print('knight moves:', bad.valid_moves(0, 1))


### Why this is painful

- **Open/Closed violated.** Adding a new piece means editing the giant `if/elif`.
- **One class knows everything.** A bug in the rook section can break pawn logic during refactors.
- **Hard to test.** You can't test "rook movement" in isolation.
- **No types.** `('P','W')` is a magic tuple - a typo like `'p'` silently breaks things.
- **No room to grow.** How would you add "is this move a check?" Another `if` pyramid.

This is the classic signal: **"if the piece type is X, do Y"** should become **polymorphism**.


## Better - One class per piece, shared interface

Each piece becomes its own class, and they all promise: *"I know my own moves."*
The `Board` becomes a dumb 8x8 grid - it doesn't know the rules, only where things are.


In [ ]:
from abc import ABC, abstractmethod

WHITE, BLACK = 'W', 'B'

class Piece(ABC):
    """Abstract base class - every piece MUST tell us its symbol and valid moves."""
    def __init__(self, color):
        self.color = color

    @abstractmethod
    def symbol(self) -> str: ...

    @abstractmethod
    def valid_moves(self, board, r, c): ...

    def __repr__(self):
        s = self.symbol()
        return s.upper() if self.color == WHITE else s.lower()


def on_board(r, c):
    return 0 <= r < 8 and 0 <= c < 8


class Rook(Piece):
    def symbol(self): return 'r'
    def valid_moves(self, board, r, c):
        moves = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            while on_board(nr, nc):
                if board[nr][nc] is None:
                    moves.append((nr, nc))
                else:
                    if board[nr][nc].color != self.color:
                        moves.append((nr, nc))  # capture
                    break
                nr += dr; nc += dc
        return moves


class Knight(Piece):
    def symbol(self): return 'n'
    def valid_moves(self, board, r, c):
        deltas = [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
        out = []
        for dr, dc in deltas:
            nr, nc = r+dr, c+dc
            if on_board(nr, nc) and (board[nr][nc] is None or board[nr][nc].color != self.color):
                out.append((nr, nc))
        return out


# Quick sanity check
board = [[None]*8 for _ in range(8)]
board[7][0] = Rook(WHITE)
board[0][1] = Knight(BLACK)
print('White rook moves:', board[7][0].valid_moves(board, 7, 0))
print('Black knight moves:', board[0][1].valid_moves(board, 0, 1))


### What just improved

- **Open/Closed:** adding a `Bishop` means writing a new class, not editing old code.
- **Single Responsibility:** each piece knows only its own rules.
- **Polymorphism:** the game code can just call `piece.valid_moves(...)` without caring which piece.
- **Typed:** `isinstance(piece, Rook)` instead of `piece[0] == 'R'`.

### Is this the Strategy pattern?

Not quite, and the difference is worth an interview minute. What we just did is
**"Replace Conditional with Polymorphism"** (Fowler) — subtype polymorphism. The
behaviour is welded to the *identity* of the object: a `Rook` moves like a rook
because it *is* a rook, forever.

**Strategy** is the composition version: the object *holds* an interchangeable
behaviour object and can be handed a different one at runtime.

```python
# Subtype polymorphism (what we built): behaviour IS the type
class Rook(Piece):
    def valid_moves(self, board, r, c): ...

# Strategy: behaviour is a field you can swap
class Piece:
    def __init__(self, color, movement):   # movement: MovementRule
        self.movement = movement
    def valid_moves(self, board, r, c):
        return self.movement.moves(board, r, c, self.color)
```

Which is right here? **Subtyping**, because a rook never becomes a bishop —
there is nothing to swap. Reach for Strategy when the *same* object must change
behaviour over its life, or when behaviours combine (see the Blackjack lab,
where a player really can be handed a different playing style each round).

Naming a pattern you did not implement is worse than naming none: it teaches the
wrong shape.


## The classes we will need (full picture)

```
+--------+        +-------+        +------+
|  Game  | has-a  | Board | 8x8 of | Cell |  0 or 1 Piece
|  turn  |------->|       |------->|      |
| history|        +-------+        +------+
+--------+
    | creates
    v
+--------+
|  Move  |   data class: from (r,c) -> to (r,c), captured piece (for undo)
+--------+

Piece (abstract)
  |-- King     |-- Queen    |-- Rook
  |-- Bishop   |-- Knight   |-- Pawn
```

**Why a separate `Move` class?**
It is a record of "what happened". We need it for:

- **Undo / takeback** - store the captured piece so we can put it back.
- **Move history** - print the game in chess notation.
- **Networking** - send a `Move` over the wire instead of the whole board.


## Mini-exercise: spot the responsibilities

Before moving on, guess where each job belongs - `Piece`, `Board`, `Game`, or `Move`.

| Job | Owner? |
|---|---|
| "Is this square inside the 8x8 grid?"              | `Board` |
| "What squares can a bishop reach from e4?"          | `Piece` (Bishop) |
| "Is it White's turn right now?"                     | `Game` |
| "Put the captured piece back on undo."              | `Move` (plus `Board`) |
| "Is the King currently in check?"                   | `Game` (asks pieces) |

If you found yourself saying "all of them", that is the exact feeling we are engineering
**away from**. In Notebook 2 we will build this - including a proper `Move` object and a
simple check detector.


## Further reading

- *Design Patterns: Elements of Reusable OO Software* (Gamma et al.) - Strategy pattern
- *Refactoring* (Fowler) - "Replace Conditional with Polymorphism"
- Robert C. Martin - **SOLID** principles, especially **S** (SRP) and **O** (OCP)
